# Arm + Sharpa Hand Interface Example

## Prerequisites

Start the C++ bridge with the joint velocity interface and the payload description of the
Sharpa hand:

```bash
fr3_joint_interface <robot_ip> fr3 velocity /home/FR3Py/fr3_bridge/config/sharpa_hand_payload.yaml
```

**Note**: If the robot is connected to another computer, make sure that the two computers are on
the same local network, and use the `tools/multicast_config.py` tool to enable multicast on both
computers:

```bash
sudo tools/multicast_config.py <interface name>
```

where interface name is the network interface that is on the same local network as the computer
connected to the robot (the computer running the real-time kernel).

Before running the notebook, make sure that:

- the FCI interface is enabled on the robot,
- the bridge above is running with the payload YAML,
- the Sharpa hand is powered and connected over USB, and its SDK is reachable.

**Important**: the pinocchio model used inside `WaypointController` still describes the Franka
gripper. `P_EE` reported by `controller.robot.getInfo` is therefore the old Franka TCP, not the
Sharpa palm. The DiffIK demo below still moves the arm correctly, but every Cartesian quantity it
reports is expressed at the Franka TCP frame.

In [ ]:
import numpy as np
from FR3Py.robot.interface import FR3Real
from FR3Py.robot.hand_interface import SharpaHand

robot = FR3Real(robot_id='fr3')
hand = SharpaHand(hand_side=1, enable_tactile=False)

arm_state = robot.getJointStates()
hand_state = hand.getJointStates()
print('arm:', arm_state)
print('hand:', hand_state)

assert hand_state is not None and hand_state['q'].shape == (22,), \
    f"expected 22 hand joint values, got {None if hand_state is None else hand_state['q'].shape}"

In [ ]:
import time

Q_MIN, Q_MAX = -0.3, 1.5
INDEX_MCP_FE = 5

q_start = hand.getJointStates()['q'].astype(np.float64)
q_lo = np.minimum(Q_MIN, q_start)
q_hi = np.maximum(Q_MAX, q_start)
print('index joints before (5-8):', q_start[5:9])

q_from = q_start[INDEX_MCP_FE]
q_to = 0.5
rate = 30.0
ramp_time = 2.0
n_steps = int(ramp_time * rate)

def send(target):
    hand.setCommands(np.clip(target, q_lo, q_hi))

for i in range(n_steps):
    alpha = (i + 1) / n_steps
    target = q_start.copy()
    target[INDEX_MCP_FE] = (1 - alpha) * q_from + alpha * q_to
    send(target)
    time.sleep(1.0 / rate)

hold = q_start.copy()
hold[INDEX_MCP_FE] = q_to
send(hold)
time.sleep(1.0)

for i in range(n_steps):
    alpha = (i + 1) / n_steps
    target = q_start.copy()
    target[INDEX_MCP_FE] = (1 - alpha) * q_to + alpha * q_from
    send(target)
    time.sleep(1.0 / rate)

send(q_start)
time.sleep(0.5)
print('index joints after (5-8):', hand.getJointStates()['q'][5:9])

Watch the hand while the cell above runs: only the index finger should move, and only at its
MCP flexion/extension joint. If any other finger or any other index joint moves, the SDK joint
ordering assumed here does not match the hardware and the index constants must be corrected before
running the rest of the notebook.

In [ ]:
import time

Q_MIN, Q_MAX = -0.3, 1.5
INDEX_MCP_FE, INDEX_MCP_AA, INDEX_PIP, INDEX_DIP = 5, 6, 7, 8
SWEEP_IDX = [INDEX_MCP_FE, INDEX_PIP, INDEX_DIP]

q_start = hand.getJointStates()['q'].astype(np.float64)
freq = 0.25
amp = 0.5
rate = 30.0
duration = 30.0

start_time = time.time()
while time.time() - start_time < duration:
    t = time.time() - start_time
    target = q_start.copy()
    target[SWEEP_IDX] = q_start[SWEEP_IDX] + amp * np.sin(2 * np.pi * freq * t)
    target[INDEX_MCP_AA] = q_start[INDEX_MCP_AA]
    hand.setCommands(np.clip(target, Q_MIN, Q_MAX))
    time.sleep(1.0 / rate)

hand.setCommands(np.clip(q_start, Q_MIN, Q_MAX))
print('sweep ended.')

In [ ]:
import time
from time import perf_counter
import numpy as np
from FR3Py.controllers.jacobianPseudoInv import WaypointController

Q_MIN, Q_MAX = -0.3, 1.5
INDEX_MCP_FE, INDEX_MCP_AA, INDEX_PIP, INDEX_DIP = 5, 6, 7, 8
SWEEP_IDX = [INDEX_MCP_FE, INDEX_PIP, INDEX_DIP]
HAND_DECIMATION = 3

controller = WaypointController(kp=1.5)
# Read the initial state of the robot
time.sleep(0.2)
state = robot.getJointStates()
hand_q_start = hand.getJointStates()['q'].astype(np.float64)
hand_target = hand_q_start.copy()
hand_call_times = []

if state is not None:
    print('starting the controller')
    q, dq = state['q'], state['dq']
    q = np.hstack([q, np.zeros(2)])
    dq = np.hstack([dq, np.zeros(2)])
    p0 = controller.robot.getInfo(q,dq)['P_EE']
    R0 = controller.robot.getInfo(q,dq)['R_EE']
    T0 = np.vstack([np.hstack([R0, p0.reshape(3,1)]), np.array([0,0,0,1])])

    freq = 0.25
    amp = 0.5
    i = 0
    start_time = time.time()
    while time.time()-start_time < 10:
        time.sleep(0.01)
        t = time.time() - start_time
        x = 0.1 * np.sin(t)
        T= T0 @ np.vstack([np.hstack([np.eye(3), np.array([0,0,x]).reshape(3,1)]), np.array([0,0,0,1])])
        state = robot.getJointStates()
        q, dq = state['q'], state['dq']
        cmd = controller.compute(q, dq, T_cmd=T)
        robot.setCommands(cmd[:7])

        if i % HAND_DECIMATION == 0:
            target = hand_q_start.copy()
            target[SWEEP_IDX] = hand_q_start[SWEEP_IDX] + amp * np.sin(2 * np.pi * freq * t)
            target[INDEX_MCP_AA] = hand_q_start[INDEX_MCP_AA]
            hand_target = np.clip(target, Q_MIN, Q_MAX)
            t_call = perf_counter()
            hand.setCommands(hand_target)
            hand_call_times.append(perf_counter() - t_call)
        i += 1

    robot.setCommands(np.zeros(7))
    hand.setCommands(hand_target)
    if hand_call_times:
        print(f'hand.setCommands: n={len(hand_call_times)} '
              f'mean={1e3*np.mean(hand_call_times):.3f} ms '
              f'max={1e3*np.max(hand_call_times):.3f} ms')
    print('Demo ended.')
else:
    print('State is None. check the connection')

In [ ]:
hand.close()